In [ ]:
import getpass, os, pathlib, subprocess, sys
from kaggle_secrets import UserSecretsClient
try:
    bearer=UserSecretsClient().get_secret('KAGGLE_TTS_TOKEN')
except Exception:
    bearer=None
if not bearer:
    bearer=getpass.getpass('输入本机已配置的服务凭据（隐藏输入）：')
if not bearer: raise ValueError('需要已有的服务凭据')
root=pathlib.Path('/kaggle/working/yice-qwen')
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(root/'env')],check=True)
runner=root/'env/bin/python'
subprocess.run([sys.executable,'-m','pip','--python',str(runner),'install','qwen-tts','faster-whisper','fastapi','uvicorn'],check=True)


In [ ]:
server_source = r'''
import base64, hashlib, io, os, secrets, threading
from collections import deque
import soundfile as sf
import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError("请在 Kaggle 设置中启用 GPU")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
token = os.environ['YICE_PRIVATE_BEARER']
app = FastAPI(docs_url=None, redoc_url=None)
gpu_lock = threading.Lock()
prompt_cache = {}
jobs_lock = threading.Lock()
jobs = {}
completed_jobs = deque()

class Request(BaseModel):
    text: list[str] = Field(min_length=1, max_length=64)
    mode: str
    ref_audio_base64: str = Field(max_length=16_000_000)
    ref_text: str
    language: str = "Chinese"
    temperature: float = Field(default=.7, gt=0, le=2)

def authorize(authorization):
    if not secrets.compare_digest(authorization.encode(), ("Bearer " + token).encode()):
        raise HTTPException(401)

def validate_request(request, authorization):
    authorize(authorization)
    if request.mode != "clone":
        raise HTTPException(400, "需要 clone 模式")
    if not request.ref_audio_base64:
        raise HTTPException(422, "参考音频无效")

def synthesize(request):
    key = hashlib.sha256((request.ref_audio_base64 + "|" + request.ref_text).encode()).hexdigest()
    with gpu_lock, torch.inference_mode():
        if key not in prompt_cache:
            try:
                audio = base64.b64decode(request.ref_audio_base64, validate=True)
                sample, rate = sf.read(io.BytesIO(audio), dtype="float32")
            except (ValueError, RuntimeError):
                raise HTTPException(422, "参考音频无效")
            prompt_cache[key] = model.create_voice_clone_prompt(
                ref_audio=(sample, rate), ref_text=request.ref_text,
            )
        waves, rate = model.generate_voice_clone(
            text=request.text, language=[request.language]*len(request.text),
            voice_clone_prompt=prompt_cache[key], temperature=request.temperature,
            max_new_tokens=min(640, max(180, max(len(t) for t in request.text) * 6)),
        )
    outputs = []
    for waveform in waves:
        buf = io.BytesIO()
        sf.write(buf, waveform, rate, format="WAV", subtype="PCM_16")
        outputs.append({"audio_base64": base64.b64encode(buf.getvalue()).decode("ascii")})
    return outputs

@app.post("/tts")
def tts(request: Request, authorization: str = Header(default="")):
    validate_request(request, authorization)
    return {"outputs": synthesize(request)}

def complete_job(job_id, request):
    try:
        result = {"status": "done", "outputs": synthesize(request)}
    except Exception as exc:
        result = {"status": "error", "detail": str(exc)}
    with jobs_lock:
        jobs[job_id] = result
        completed_jobs.append(job_id)
        if len(completed_jobs) > 32:
            del jobs[completed_jobs.popleft()]

@app.post("/tts_async")
def tts_async(request: Request, authorization: str = Header(default="")):
    validate_request(request, authorization)
    with jobs_lock:
        job_id = secrets.token_hex(8)
        jobs[job_id] = {"status": "running"}
    worker = threading.Thread(target=complete_job, args=(job_id, request), daemon=True)
    try:
        worker.start()
    except RuntimeError:
        with jobs_lock:
            del jobs[job_id]
        raise
    return {"job_id": job_id, "status": "running"}

@app.get("/tts_job/{job_id}")
def tts_job(job_id: str, authorization: str = Header(default="")):
    authorize(authorization)
    with jobs_lock:
        result = jobs.get(job_id)
    if result is None:
        raise HTTPException(404, "任务不存在或已过期")
    return result

class Recognition(BaseModel):
    audio: str = Field(max_length=64_000_000)

@app.post('/asr')
def recognize(request: Recognition, authorization: str = Header(default='')):
    authorize(authorization)
    from faster_whisper import WhisperModel
    with gpu_lock:
        if not hasattr(recognize,'model'):
            from ctypes import CDLL
            try:
                CDLL('libcublas.so.12')
                device,precision='cuda','int8_float16'
            except OSError:
                device,precision='cpu','int8'
            recognize.model=WhisperModel('large-v3',device=device,
                        device_index=int(os.getenv('YICE_ASR_DEVICE','0')) if device=='cuda' else 0,
                        compute_type=precision,cpu_threads=4)
        import math
        from scipy.signal import resample_poly
        data,rate=sf.read(io.BytesIO(base64.b64decode(request.audio,validate=True)),dtype='float32')
        if data.ndim==2: data=data.mean(axis=1)
        if rate!=16000:
            factor=math.gcd(rate,16000)
            data=resample_poly(data,16000//factor,rate//factor).astype('float32')
        segments,_=recognize.model.transcribe(data,language='zh',beam_size=5,word_timestamps=True,
                                              condition_on_previous_text=False)
        rows=[dict(text=s.text,start=s.start,end=s.end,
                   words=[dict(word=w.word,start=w.start,end=w.end) for w in s.words or []]) for s in segments]
    return dict(text=''.join(s['text'] for s in rows),segments=rows)


import pathlib, platform, re, socket, subprocess, time, urllib.request
import uvicorn
binary = pathlib.Path("/kaggle/working/cloudflared")
if not binary.exists():
    arch = {"x86_64":"amd64", "aarch64":"arm64"}[platform.machine()]
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-"+arch, binary)
    binary.chmod(0o700)
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, access_log=False))
threading.Thread(target=server.run, daemon=True).start()
tunnel = subprocess.Popen([str(binary), "tunnel", "--url", f"http://127.0.0.1:{port}"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def watch_tunnel():
    for line in tunnel.stdout:
        match = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
        if match and not pathlib.Path(os.environ["YICE_PRIVATE_ENDPOINT_FILE"]).exists():
            fd = os.open(os.environ["YICE_PRIVATE_ENDPOINT_FILE"], os.O_CREAT|os.O_EXCL|os.O_WRONLY, 0o600)
            with os.fdopen(fd, "w") as output:
                output.write(match.group(0)+"/tts")
threading.Thread(target=watch_tunnel, daemon=True).start()
while not server.should_exit: time.sleep(5)
'''


In [ ]:
import time
server_file=root/'yice_voice_service.py'
server_file.write_text(server_source)
endpoint_file=root/'private-endpoint'
endpoint_file.unlink(missing_ok=True)
service_log=root/'voice-service.log'
with service_log.open('w') as stream:
    service=subprocess.Popen([str(runner),'-u',str(server_file)],env={**os.environ,'YICE_PRIVATE_BEARER':bearer,'YICE_PRIVATE_ENDPOINT_FILE':str(endpoint_file),
                             'CUDA_VISIBLE_DEVICES':'0,1','YICE_ASR_DEVICE':'1'},stdout=stream,stderr=subprocess.STDOUT)
for _ in range(360):
    if endpoint_file.is_file() and endpoint_file.stat().st_size:
        from IPython.display import display, HTML
        import html
        display(HTML('<span>本次私有端点（仅复制到本机配置）</span><code>' +
                     html.escape(endpoint_file.read_text()) + '</code>'))
        break
    if service.poll() is not None: raise RuntimeError('私有语音服务启动失败，请查看无凭据的服务日志')
    time.sleep(1)
else:
    raise TimeoutError('服务启动超时')
